# 项目五：双智能体协作小组 — Multi-Agent（多智能体）

在前四个项目中，我们已经掌握了单个智能体的各种能力：
- 项目一：Prompt Engineering（提示词工程）
- 项目二：Tool Use（工具调用）
- 项目三：Memory（记忆系统）
- 项目四：Code Sandbox（代码沙箱）

但现实中，很多复杂任务**不是一个"全能选手"能搞定的**。

> 就像一家公司：产品经理负责拆解需求，程序员负责写代码，测试员负责找 bug——**分工协作**才能高效完成。

本项目将学习：

1. 理解 **Multi-Agent（多智能体）** 的核心思想：让多个 AI 角色分工合作
2. 掌握智能体之间的 **消息传递** 和 **任务流转** 机制
3. 学会设计 **角色分工** 和 **协作流程**
4. 构建一个"产品经理 + 程序员"的双智能体协作系统

---

## 一、什么是 Multi-Agent？

### 1.1 为什么需要多个智能体？

单个智能体的问题：

| 问题 | 说明 |
|------|------|
| 角色混乱 | 一个模型又要写文案、又要写代码、又要审核，容易"分心" |
| 质量下降 | 没有"第二双眼睛"检查，错误容易被忽略 |
| 难以扩展 | 任务越复杂，单个 Prompt 越难覆盖所有需求 |

多智能体的优势：

| 优势 | 说明 |
|------|------|
| **专业分工** | 每个智能体专注一个角色，System Prompt 更精准 |
| **互相检查** | 一个智能体的输出可以被另一个智能体审核 |
| **模块化** | 可以独立替换、升级某个智能体，不影响其他 |
| **模拟真实流程** | 像公司一样，按流程协作（需求 → 开发 → 测试 → 交付） |

### 1.2 核心概念

```
┌──────────────────────────────────────────────────────────┐
│  Multi-Agent 系统的三个核心要素：                          │
│                                                          │
│  1. 角色（Role）                                         │
│     每个智能体有自己的身份、职责和 System Prompt            │
│     例：产品经理、程序员、测试员、文案编辑                   │
│                                                          │
│  2. 消息传递（Message Passing）                           │
│     智能体之间通过"消息"通信                               │
│     例：产品经理把需求文档"发给"程序员                      │
│                                                          │
│  3. 协作流程（Workflow）                                  │
│     定义智能体的执行顺序和数据流向                          │
│     例：需求 → 开发 → 审核 → 交付                         │
└──────────────────────────────────────────────────────────┘
```

### 1.3 常见的协作模式

| 模式 | 说明 | 适用场景 |
|------|------|----------|
| **串行流水线** | A → B → C，依次处理 | 需求分析 → 编码 → 测试 |
| **主从模式** | 一个"主管"分配任务给多个"下属" | 项目经理分配任务给团队成员 |
| **辩论模式** | 两个智能体互相讨论、反驳 | 方案评审、决策分析 |
| **投票模式** | 多个智能体各自回答，选最优 | 需要高可靠性的决策 |

本项目重点学习**串行流水线**和**辩论模式**。

---

## 二、基础准备

### 2.1 导入依赖

本项目继续使用 DeepSeek API，沿用之前的基础设施：

In [ ]:
# Ensure openai is installed in JupyterLite/Pyodide before importing it.
try:
    from openai import OpenAI
except ModuleNotFoundError:
    import micropip
    await micropip.install("openai")
    from openai import OpenAI

import os

# 初始化 DeepSeek 客户端
api_key = os.environ.get("DEEPSEEK_API_KEY", "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

def call_llm(system_prompt, user_message, temperature=0.7):
    """
    调用 DeepSeek API 的通用函数。
    
    参数:
        system_prompt: 系统提示词（定义智能体角色）
        user_message: 用户消息（输入内容）
        temperature: 温度参数
    
    返回:
        str: 模型的回复内容
    """
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

print("DeepSeek API 连接成功！")

### 2.2 定义智能体类

在多智能体系统中，每个智能体都有自己的角色和行为。我们封装一个通用的 `Agent` 类：

```python
class Agent:
    def __init__(self, name, system_prompt):
        self.name = name          # 智能体名称
        self.system_prompt = system_prompt  # 角色定义
    
    def run(self, task):
        """接收任务，返回结果"""
        return call_llm(self.system_prompt, task)
```

**关键点：**

| 属性 | 说明 |
|------|------|
| `name` | 智能体的名字，用于日志和调试 |
| `system_prompt` | 定义智能体的角色、职责和行为规范 |
| `run(task)` | 接收输入，调用 LLM，返回输出 |

**多智能体的本质就是：一个智能体的输出，作为另一个智能体的输入。**

---

## 三、示例一：串行流水线 — "产品经理 + 程序员" 协作

### 3.1 场景

用户提出一个产品想法，系统自动完成：

```
用户想法 → 产品经理（拆解需求） → 程序员（编写代码） → 输出结果
```

### 3.2 协作流程图

```
┌──────────┐     需求文档      ┌──────────┐     代码结果      ┌──────────┐
│ 产品经理  │ ──────────────→ │  程序员   │ ──────────────→ │  最终输出 │
│ PM_Agent  │                  │ Dev_Agent │                  │          │
└──────────┘                  └──────────┘                  └──────────┘
     ↑                              │
     │                              │
  用户想法                      编写代码
```

### 3.3 代码实现

In [ ]:
class Agent:
    """通用智能体基类"""
    
    def __init__(self, name, system_prompt):
        self.name = name
        self.system_prompt = system_prompt
    
    def run(self, task):
        """接收任务，返回结果"""
        print(f"  [{self.name}] 正在处理...")
        result = call_llm(self.system_prompt, task, temperature=0.7)
        return result


# ===== 定义两个智能体 =====

# 产品经理：负责拆解需求，输出结构化的需求文档
pm_agent = Agent(
    name="产品经理",
    system_prompt="""你是一个资深产品经理。你的职责是：
1. 接收用户的产品想法
2. 分析核心需求，拆解为具体的功能点
3. 输出一份结构化的技术需求文档，包含：
   - 产品概述（一句话描述）
   - 核心功能列表（编号列出）
   - 技术实现建议（用什么语言/库）
   - 输入输出示例

请用 Markdown 格式输出，确保程序员能直接根据你的文档编写代码。"""
)

# 程序员：负责根据需求文档编写代码
dev_agent = Agent(
    name="程序员",
    system_prompt="""你是一个 Python 编程专家。你的职责是：
1. 接收产品经理的需求文档
2. 根据文档中的功能列表，编写完整的 Python 代码
3. 代码要求：
   - 有清晰的函数/类结构
   - 有必要的注释
   - 包含测试用例（用 print 输出结果）
4. 只返回代码（用 ```python 包裹），不要多余解释"""
)


# ===== 构建流水线 =====

def software_pipeline(user_idea):
    """软件开发流水线：产品经理 → 程序员"""
    print("=" * 60)
    print(f"用户需求：{user_idea}")
    print("=" * 60)
    
    # 第 1 步：产品经理拆解需求
    print("\n--- 第 1 步：产品经理拆解需求 ---")
    spec_doc = pm_agent.run(user_idea)
    print(f"\n{spec_doc}")
    
    # 第 2 步：程序员根据需求文档编写代码
    print("\n--- 第 2 步：程序员编写代码 ---")
    code_result = dev_agent.run(f"请根据以下需求文档编写代码：\n\n{spec_doc}")
    print(f"\n{code_result}")
    
    print("\n" + "=" * 60)
    print("开发完成！")
    return spec_doc, code_result


# ===== 测试 =====
software_pipeline("一个命令行版的待办事项管理器，支持添加、删除、查看和标记完成")

### 3.4 运行分析

运行上面的代码，你会看到：

1. **产品经理** 把"待办事项管理器"拆解为具体的功能点和技术方案
2. **程序员** 根据需求文档，生成完整的 Python 代码

**注意观察：** 程序员生成的代码质量，很大程度取决于产品经理的需求文档质量。这就是多智能体系统中"上游影响下游"的特点。

---

## 四、示例二：辩论模式 — "正方 + 反方" 方案评审

### 4.1 场景

面对一个决策问题，让两个智能体分别扮演正方和反方，进行多轮辩论，最后由"裁判"总结。

```
┌──────────┐                    ┌──────────┐
│  正方     │ ←─── 互相反驳 ───→ │  反方     │
│ For_Agent │                    │ Against  │
└──────────┘                    └──────────┘
      │                               │
      └───────────┬───────────────────┘
                  ↓
            ┌──────────┐
            │  裁判     │
            │ Judge    │
            └──────────┘
```

### 4.2 代码实现

In [ ]:
# ===== 定义三个智能体 =====

for_agent = Agent(
    name="正方",
    system_prompt="""你是一个辩论赛的正方辩手。你的职责是：
1. 为给定的辩题进行正面论证
2. 提出 2-3 个有力的论点和论据
3. 如果收到反方的反驳，要针对性地回应
4. 保持逻辑严密，用事实和数据说话
5. 每次发言控制在 200 字以内"""
)

against_agent = Agent(
    name="反方",
    system_prompt="""你是一个辩论赛的反方辩手。你的职责是：
1. 为给定的辩题进行反面论证
2. 提出 2-3 个有力的论点和论据
3. 如果收到正方的论点，要针对性地反驳
4. 保持逻辑严密，用事实和数据说话
5. 每次发言控制在 200 字以内"""
)

judge_agent = Agent(
    name="裁判",
    system_prompt="""你是一个辩论赛的裁判。你的职责是：
1. 听取正方和反方的辩论
2. 客观评价双方的论点质量
3. 指出双方的亮点和不足
4. 给出你的最终判断和总结
5. 用 Markdown 格式输出评审报告"""
)


# ===== 辩论流程 =====

def debate(topic, rounds=2):
    """
    辩论模式：正方和反方进行多轮辩论，裁判最后总结。
    
    参数:
        topic: 辩题
        rounds: 辩论轮数
    """
    print("=" * 60)
    print(f"辩题：{topic}")
    print(f"辩论轮数：{rounds}")
    print("=" * 60)
    
    debate_history = f"辩题：{topic}\n"
    
    for i in range(1, rounds + 1):
        print(f"\n{'─'*40}")
        print(f"第 {i} 轮辩论")
        print(f"{'─'*40}")
        
        # 正方发言
        if i == 1:
            for_input = f"辩题是：{topic}。请作为正方进行立论。"
        else:
            for_input = f"辩题是：{topic}。\n\n之前的辩论记录：\n{debate_history}\n\n请针对反方的观点进行反驳，并强化你的论点。"
        
        for_statement = for_agent.run(for_input)
        print(f"\n【正方发言】\n{for_statement}")
        debate_history += f"\n【正方第{i}轮】\n{for_statement}\n"
        
        # 反方发言
        against_input = f"辩题是：{topic}。\n\n之前的辩论记录：\n{debate_history}\n\n请针对正方的观点进行反驳，并强化你的论点。"
        against_statement = against_agent.run(against_input)
        print(f"\n【反方发言】\n{against_statement}")
        debate_history += f"\n【反方第{i}轮】\n{against_statement}\n"
    
    # 裁判总结
    print(f"\n{'─'*40}")
    print("裁判评审")
    print(f"{'─'*40}")
    
    judge_input = f"以下是完整的辩论记录：\n{debate_history}\n\n请进行评审总结。"
    verdict = judge_agent.run(judge_input)
    print(f"\n【裁判总结】\n{verdict}")
    
    return verdict


# ===== 测试 =====
debate("人工智能是否会取代程序员？", rounds=2)

### 4.3 运行分析

运行上面的代码，你会看到一场精彩的辩论：

1. **第 1 轮**：正方立论 → 反方立论
2. **第 2 轮**：正方反驳反方 → 反方反驳正方
3. **裁判总结**：客观评价双方论点，给出最终判断

**关键设计：**

| 设计点 | 说明 |
|--------|------|
| `debate_history` | 记录所有发言，让每轮都能参考之前的内容 |
| 第 1 轮 vs 后续轮 | 第 1 轮是立论，后续轮是反驳 |
| 裁判独立 | 裁判不参与辩论，只做最终评审 |

**两种模式的对比：**

| | 示例一（串行流水线） | 示例二（辩论模式） |
|---|---|---|
| 数据流向 | 单向：A → B | 双向：A ↔ B |
| 执行次数 | 每个智能体执行 1 次 | 每个智能体执行多轮 |
| 适用场景 | 有明确先后顺序的任务 | 需要多角度分析的问题 |

---

## 五、课后练习

### 练习 1（基础）：文案创作流水线

**目标：** 构建一个"营销总监 + 文案写手"的串行流水线。营销总监负责分析产品卖点和目标受众，文案写手根据分析结果撰写广告文案。

**要求：**
1. 定义 `marketing_agent`（营销总监），System Prompt 要求它分析：
   - 产品核心卖点
   - 目标用户画像
   - 文案风格建议
   - 推荐的文案方向（3 个）
2. 定义 `copywriter_agent`（文案写手），System Prompt 要求它根据营销总监的分析撰写文案
3. 补全 `ad_pipeline` 函数，串联两个智能体

**提示：**
- 参考示例一的 `software_pipeline` 实现
- 营销总监的输出是文案写手的输入
- 文案写手应该输出 3 条不同风格的广告语

**测试用例：**
- 产品 1："一款面向年轻人的智能手表，主打运动监测和社交功能，售价 599 元"
- 产品 2："一家新开的宠物咖啡馆，提供撸猫撸狗和精品咖啡，人均消费 50 元"

In [ ]:
# ===== 练习 1：文案创作流水线 =====

# 第一步：定义营销总监
marketing_agent = Agent(
    name="营销总监",
    system_prompt="""TODO: 补全营销总监的角色定义
    要求：
    - 分析产品核心卖点
    - 分析目标用户画像
    - 给出文案风格建议
    - 推荐 3 个文案方向
    """
)

# 第二步：定义文案写手
copywriter_agent = Agent(
    name="文案写手",
    system_prompt="""TODO: 补全文案写手的角色定义
    要求：
    - 根据营销总监的分析撰写文案
    - 输出 3 条不同风格的广告语
    - 每条广告语标注风格（如：走心、幽默、理性）
    """
)


# 第三步：构建流水线
def ad_pipeline(product_info):
    """广告文案流水线：营销总监 → 文案写手"""
    # TODO: 1. 营销总监分析产品
    # TODO: 2. 文案写手根据分析结果撰写文案
    # TODO: 3. 返回最终文案
    pass


# ===== 测试 =====
print("=== 测试 1：智能手表 ===")
ad_pipeline("一款面向年轻人的智能手表，主打运动监测和社交功能，售价 599 元")

print("\n\n=== 测试 2：宠物咖啡馆 ===")
ad_pipeline("一家新开的宠物咖啡馆，提供撸猫撸狗和精品咖啡，人均消费 50 元")

### 练习 2（进阶）：代码审查流水线

**目标：** 构建一个"程序员 + 代码审查员 + 总结者"的三级流水线。程序员写代码，审查员找问题，总结者输出最终报告。

**场景：**

```
用户需求 → 程序员（写代码） → 审查员（审查代码） → 总结者（输出报告）
```

**要求：**
1. 定义 `coder_agent`（程序员），根据需求编写 Python 代码
2. 定义 `reviewer_agent`（审查员），审查代码并给出修改建议，包括：
   - 代码质量问题（命名、结构）
   - 潜在 bug
   - 性能建议
   - 安全评分（1-10 分）
3. 定义 `summarizer_agent`（总结者），汇总代码和审查意见，输出最终报告
4. 补全 `code_review_pipeline` 函数，串联三个智能体

**提示：**
- 程序员的输出是审查员的输入
- 审查员的输出和原始代码一起传给总结者
- 总结者需要输出：最终代码 + 审查摘要 + 改进建议

**测试用例：**
- 需求 1："写一个 Python 函数，实现一个简单的计算器，支持加减乘除"
- 需求 2："写一个 Python 程序，统计一段文本中每个词出现的次数"

In [ ]:
# ===== 练习 2：代码审查流水线 =====

# 第一步：定义三个智能体
coder_agent = Agent(
    name="程序员",
    system_prompt="""TODO: 补全程序员的角色定义
    要求：根据用户需求编写完整的 Python 代码，包含注释和测试用例
    """
)

reviewer_agent = Agent(
    name="代码审查员",
    system_prompt="""TODO: 补全代码审查员的角色定义
    要求：
    - 审查代码质量（命名、结构、注释）
    - 找出潜在 bug
    - 给出性能建议
    - 给出安全评分（1-10 分）
    """
)

summarizer_agent = Agent(
    name="总结者",
    system_prompt="""TODO: 补全总结者的角色定义
    要求：
    - 汇总原始代码和审查意见
    - 输出改进后的代码
    - 输出审查摘要
    - 输出最终建议
    """
)


# 第二步：构建三级流水线
def code_review_pipeline(requirement):
    """代码审查流水线：程序员 → 审查员 → 总结者"""
    # TODO: 1. 程序员根据需求写代码
    # TODO: 2. 审查员审查代码
    # TODO: 3. 总结者汇总代码和审查意见，输出最终报告
    # TODO: 4. 返回最终报告
    pass


# ===== 测试 =====
print("=== 测试 1：计算器 ===")
code_review_pipeline("写一个 Python 函数，实现一个简单的计算器，支持加减乘除")

print("\n\n=== 测试 2：词频统计 ===")
code_review_pipeline("写一个 Python 程序，统计一段文本中每个词出现的次数")

---
## 小结

恭喜你完成了全部五个项目！回顾一下本项目的核心知识点：

| 知识点 | 要点 |
|--------|------|
| Multi-Agent 的本质 | 多个专业智能体分工协作，比单个"全能"智能体更可靠 |
| 三个核心要素 | 角色（System Prompt）、消息传递（输出→输入）、协作流程 |
| 串行流水线 | A → B → C，上游输出是下游输入 |
| 辩论模式 | A ↔ B 多轮交锋，裁判总结 |
| Agent 类封装 | 每个智能体有 name、system_prompt、run() 方法 |

**五个项目知识图谱：**

```
项目一：Prompt Engineering ──→ 智能体的"大脑"
项目二：Tool Use ────────────→ 智能体的"手和眼"
项目三：Memory ──────────────→ 智能体的"记忆"
项目四：Code Sandbox ────────→ 智能体的"执行力"
项目五：Multi-Agent ─────────→ 智能体的"团队协作"
```

掌握了这五大核心能力，你就具备了构建复杂 AI 智能体应用的基础！